In [1]:
import operator
from typing import Annotated, Literal, TypedDict
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

# Load environment variables
load_dotenv()

# Initialize Groq LLMs
generator_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)
evaluator_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
optimizer_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)


# 1. Define Pydantic Schema for Structured Output
class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(
        ..., description="Final evaluation result."
    )
    feedback: str = Field(..., description="Feedback for the tweet.")


structured_evaluator_llm = evaluator_llm.with_structured_output(
    TweetEvaluation
)


# 2. Define State Schema
class TweetState(TypedDict):
    topic: str
    tweet: str
    evaluation: Literal["approved", "needs_improvement"]
    feedback: str
    iteration: int
    max_iteration: int

    tweet_history: Annotated[list[str], operator.add]
    feedback_history: Annotated[list[str], operator.add]


# 3. Define Node Functions
def generate_tweet(state: TweetState):
    messages = [
        SystemMessage(
            content="You are a funny and clever Twitter/X influencer."
        ),
        HumanMessage(
            content=(
                f"Write a short, original, and hilarious tweet on the topic: \"{state['topic']}\".\n"
                "Rules:\n"
                "- Do NOT use question-answer format.\n"
                "- Max 280 characters.\n"
                "- Use observational humor, irony, sarcasm, or cultural references.\n"
                "- Think in meme logic, punchlines, or relatable takes.\n"
                "- Use simple, day-to-day English."
            )
        ),
    ]

    response = generator_llm.invoke(messages).content
    return {"tweet": response, "tweet_history": [response]}


def evaluate_tweet(state: TweetState):
    messages = [
        SystemMessage(
            content=(
                "You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets "
                "based on humor, originality, virality, and tweet format."
            )
        ),
        HumanMessage(
            content=(
                f"Evaluate the following tweet:\nTweet: \"{state['tweet']}\"\n\n"
                "Use the criteria below to evaluate the tweet:\n"
                "1. Originality – Is this fresh, or have you seen it a hundred times before?\n"
                "2. Humor – Did it genuinely make you smile, laugh, or chuckle?\n"
                "3. Punchiness – Is it short, sharp, and scroll-stopping?\n"
                "4. Virality Potential – Would people retweet or share it?\n"
                "5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?\n\n"
                "Auto-reject if:\n"
                "- It's written in question-answer format (e.g., 'Why did...' or 'What happens when...')\n"
                "- It exceeds 280 characters\n"
                "- It reads like a traditional setup-punchline joke\n"
                "- Ends with generic, throwaway, or deflating lines that weaken the humor\n\n"
                "Respond ONLY in structured format:\n"
                "- evaluation: 'approved' or 'needs_improvement'\n"
                "- feedback: One paragraph explaining the strengths and weaknesses"
            )
        ),
    ]

    response = structured_evaluator_llm.invoke(messages)

    return {
        "evaluation": response.evaluation,
        "feedback": response.feedback,
        "feedback_history": [response.feedback],
    }


def optimize_tweet(state: TweetState):
    messages = [
        SystemMessage(
            content="You punch up tweets for virality and humor based on given feedback."
        ),
        HumanMessage(
            content=(
                f"Improve the tweet based on this feedback:\n\"{state['feedback']}\"\n\n"
                f"Topic: \"{state['topic']}\"\n"
                f"Original Tweet: {state['tweet']}\n\n"
                "Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters."
            )
        ),
    ]

    response = optimizer_llm.invoke(messages).content
    iteration = state["iteration"] + 1

    return {
        "tweet": response,
        "iteration": iteration,
        "tweet_history": [response],
    }


# 4. Define Conditional Edge Logic
def route_evaluation(state: TweetState):
    if (
        state["evaluation"] == "approved"
        or state["iteration"] >= state["max_iteration"]
    ):
        return "approved"
    else:
        return "needs_improvement"


# 5. Build Graph
graph = StateGraph(TweetState)

# Add Nodes
graph.add_node("generate", generate_tweet)
graph.add_node("evaluate", evaluate_tweet)
graph.add_node("optimize", optimize_tweet)

# Add Edges
graph.add_edge(START, "generate")
graph.add_edge("generate", "evaluate")
graph.add_conditional_edges(
    "evaluate",
    route_evaluation,
    {"approved": END, "needs_improvement": "optimize"},
)
graph.add_edge("optimize", "evaluate")

# Compile Workflow
workflow = graph.compile()

# 6. Execute Workflow
initial_state = {
    "topic": "Remote work fatigue",
    "iteration": 1,
    "max_iteration": 5,
}

result = workflow.invoke(initial_state)

# Display Output
print("--- FINAL RESULT ---")
print(f"Evaluation: {result['evaluation']}")
print(f"Final Tweet: {result['tweet']}\n")

print("--- TWEET ITERATIONS HISTORY ---")
for idx, tweet in enumerate(result["tweet_history"], start=1):
    print(f"Version {idx}: {tweet}")

--- FINAL RESULT ---
Evaluation: approved
Final Tweet: "Remote work fatigue is real. I've been in PJs for 3 days and my cat thinks I'm a roommate who doesn't pay rent" #remotework

--- TWEET ITERATIONS HISTORY ---
Version 1: "Remote work fatigue is real. I've been in PJs for 3 days and my cat thinks I'm a roommate who doesn't pay rent" #remotework
